### Olist Silver ELT Scripts - NB 1


In [ ]:
# This notebook performs ELT/ETL operations on dimensional related tables in the
# olist datasets. The data is ingested from the bronze layer tables, cleaned, deduped and
# loaded to tables in the silver layer ready for data modelling and analysis

# Developer: Asif Shah
# Created Date: 16.09.2026

In [2]:
# Import all required pyspark libraries and objects

from pyspark.sql import functions as f
from pyspark.sql.types import IntegerType, DecimalType  
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Enable schema evolution for silver dimensional data:

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

StatementMeta(, 99eeaab4-27b5-453f-857a-6556cdfbd485, 4, Finished, Available, Finished, False)

In [ ]:
# Product data transformation

# 1. Read Bronze table(s)

df_olist_products = spark.read.format("delta").table("Bronze.olist_products")

# 2. Deduplicate, clean and load

df_cleanup = (df_olist_products
    .drop_duplicates(["product_id"]) 
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
    .withColumn("product_width_cm", f.col("product_width_cm").cast(DecimalType(10, 4)))
    .withColumn("product_name_length", f.col("product_name_length").cast(IntegerType()))
    .withColumn("product_photos_qty", f.col("product_photos_qty").cast(IntegerType()))
    .withColumn("product_description_length", f.col("product_description_length").cast(IntegerType()))
    .withColumn("product_weight_g", f.col("product_weight_g").cast(IntegerType()))
    .withColumn("product_length_cm", f.col("product_length_cm").cast(DecimalType(10, 4)))
    .withColumn("product_height_cm", f.col("product_height_cm").cast(DecimalType(10, 4)))
    
    .fillna({"product_category_name":"Unknown", 
             "product_width_cm":0.0, 
             "product_name_length":0, 
              "product_weight_g": 0, 
             "product_photos_qty":0,
             "product_height_cm":0.0, 
            "product_length_cm": 0.0, 
             "product_description_length": 0})

    .withColumn("product_category_name", f.initcap(f.regexp_replace(f.col("product_category_name"),"_"," ")))
    .withColumn("silver_ingested_at", f.current_timestamp())
    .withColumn("silver_updated_at", f.current_timestamp())
    .drop("ingestion_time")
)

# 3. view the final result - test before loading then comment out
# display(df_cleanup)

# 4. Create table if not already exists
if not spark.catalog.tableExists("Silver.olist_products"):
    df_cleanup.write.format("delta").saveAsTable("Silver.olist_products")



In [8]:
# product data merge into silver from bronze

silver_prod_tbl = DeltaTable.forName(spark, "Silver.olist_products")

silver_prod_tbl.alias("target") \
  .merge (
    source = df_cleanup.alias("source") ,
    condition = "target.product_id = source.product_id"
          ) \
  .whenMatchedUpdate(
   set = {
        "product_category_name" : "source.product_category_name",
        "product_name_length" : "source.product_name_length",
        "product_description_length" : "source.product_description_length",
        "product_photos_qty": "source.product_photos_qty",
        "product_weight_g" : "source.product_weight_g",
        "product_length_cm": "source.product_length_cm",
        "product_height_cm" : "source.product_height_cm",
        "product_width_cm" : "source.product_width_cm" , 
        "silver_updated_at": f.current_timestamp()
   }
  ) \
  .whenNotMatchedInsertAll() \
  .execute()
  


StatementMeta(, fdb79b96-70c1-4d09-906d-e8aa5cfc9316, 10, Finished, Available, Finished, False)

In [4]:
# olist customer transformation

# 1. Read Bronze table(s)

df_olist_customers = spark.read.format("delta").table("Bronze.olist_customers")

# 2. Deduplicate, clean and load

df_cust_cleanup = (df_olist_customers 
    .drop_duplicates(subset=["customer_id", "customer_unique_id"])

    .fillna({"customer_zip_code_prefix":"Unknown", 
             "customer_city":"Unknown", 
             "customer_state": "Unknown" })

    .withColumn("customer_city", f.initcap(f.col("customer_city")))
    .withColumn("silver_ingested_at", f.current_timestamp())
    .withColumn("silver_updated_at", f.current_timestamp())
    .drop("Ingestion_Time")
    
)

# 3. view the final result - test before loading then comment out
# display(df_cust_cleanup)

# 4. Create table if not already exists
if not spark.catalog.tableExists("Silver.olist_customers"):
    df_cust_cleanup.write.format("delta").saveAsTable("Silver.olist_customers")

StatementMeta(, 99eeaab4-27b5-453f-857a-6556cdfbd485, 6, Finished, Available, Finished, False)

In [6]:
# customer data merge into silver from bronze

# disable schema automerge for this table as its possible that it can contain PII data
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")

silver_cust_tbl = DeltaTable.forName(spark, "Silver.olist_customers")

silver_cust_tbl.alias("target") \
  .merge (
    source = df_cust_cleanup.alias("source") ,
    condition = "target.customer_id = source.customer_id AND target.customer_unique_id = source.customer_unique_id"
                
          ) \
  .whenMatchedUpdate(
   set = {
        "customer_zip_code_prefix" : "source.customer_zip_code_prefix",
        "customer_city" : "source.customer_city",
        "customer_state" : "source.customer_state",
        "silver_updated_at": f.current_timestamp()
   }
  ) \
  .whenNotMatchedInsertAll() \
  .execute()

# re-enable schema automerge for other tables 
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

StatementMeta(, 99eeaab4-27b5-453f-857a-6556cdfbd485, 8, Finished, Available, Finished, False)

In [10]:
# olist seller transformation

# 1. Read Bronze table(s)

df_olist_sellers = spark.read.format("delta").table("Bronze.olist_sellers")

# 2. Deduplicate, clean and load

df_sellers_cleanup = (df_olist_sellers 
    .drop_duplicates(["seller_id"])

    .fillna({"seller_zip_code_prefix":"Unknown", 
             "seller_city":"Unknown", 
             "seller_state": "Unknown" })

    .withColumn("seller_city", f.initcap(f.col("seller_city")))
    .withColumn("silver_ingested_at", f.current_timestamp())
    .withColumn("silver_updated_at", f.current_timestamp())
    .drop("ingestion_time")
    
)

# 3. view the final result - test before loading then comment out
# display(df_sellers_cleanup)

# 4. Create table if not already exists
if not spark.catalog.tableExists("Silver.olist_sellers"):
    df_sellers_cleanup.write.format("delta").saveAsTable("Silver.olist_sellers")

StatementMeta(, 99eeaab4-27b5-453f-857a-6556cdfbd485, 12, Finished, Available, Finished, False)

In [11]:
# seller data merge into silver from bronze

silver_seller_tbl = DeltaTable.forName(spark, "Silver.olist_sellers")

silver_seller_tbl.alias("target") \
  .merge (
    source = df_sellers_cleanup.alias("source") ,
    condition = "target.seller_id = source.seller_id"
                
          ) \
  .whenMatchedUpdate(
   set = {
        "seller_zip_code_prefix" : "source.seller_zip_code_prefix",
        "seller_city" : "source.seller_city",
        "seller_state" : "source.seller_state",
        "silver_updated_at": f.current_timestamp()
   }
  ) \
  .whenNotMatchedInsertAll() \
  .execute()

StatementMeta(, 99eeaab4-27b5-453f-857a-6556cdfbd485, 13, Finished, Available, Finished, False)

In [ ]:
# olist geolocation transformation


# 1. Read Bronze table
df_olist_geoloc = spark.read.format("delta").table("Bronze.olist_geolocation")

# 2. Create the Window 
window_spec = Window.partitionBy("geolocation_zip_code_prefix").orderBy(f.col("geolocation_lat").desc())

# 3. Apply window AND filter in a single pipeline stream to avoid data skew
silver_geoloc = (df_olist_geoloc
    .withColumn("row_num", f.row_number().over(window_spec))
    .filter(f.col("row_num") == 1) # Drops the millions of duplicates BEFORE display/write
    .drop("row_num")
)

final_geoloc_df = silver_geoloc.coalesce(4)

# 4. Base Clean and Format (Do this first so the window processes clean data)
df_geo_cleanup = (final_geoloc_df
    .fillna({"geolocation_city": "Unknown", "geolocation_state": "Unknown"})
    .withColumn("geolocation_city", f.initcap(f.col("geolocation_city")))
    .withColumn("geolocation_lat", f.round(f.col("geolocation_lat"), 4).cast(DecimalType(8, 4)))
    .withColumn("geolocation_lng", f.round(f.col("geolocation_lng"), 4).cast(DecimalType(8, 4)))
    .withColumn("silver_ingested_at", f.current_timestamp())
    .withColumn("silver_updated_at", f.current_timestamp())
    .drop("Ingestion_Time")
)

# 5. view the final result - test before loading then comment out
# display(df_geo_cleanup)

# 6. Create table if not already exists
if not spark.catalog.tableExists("Silver.olist_geolocation"):
    df_geo_cleanup.write.format("delta").saveAsTable("Silver.olist_geolocation")


In [ ]:
# geolocation data merge into silver from bronze

silver_geoloc_tbl = DeltaTable.forName(spark, "Silver.olist_geolocation")


silver_geoloc_tbl.alias("target") \
  .merge (
    source = df_geo_cleanup.alias("source") ,
    condition = "target.geolocation_zip_code_prefix = source.geolocation_zip_code_prefix"       
          ) \
  .whenMatchedUpdate(
   set = {
        "geolocation_lat" : "source.geolocation_lat",
        "geolocation_lng" : "source.geolocation_lng",
        "geolocation_city" : "source.geolocation_city",
        "geolocation_state" : "source.geolocation_state",
        "silver_updated_at" : f.current_timestamp()}
  ) \
  .whenNotMatchedInsertAll() \
  .execute() 



In [ ]:
# order table cleanup and load from bronze to silver

# from pyspark.sql import functions as f 

# 1. read bronze table(s)
df_olist_orders = spark.read.format("delta").table("Bronze.olist_orders")

# 2. deduplicate, correct data types and clean data
df_ord_cleanup = (df_olist_orders
    .drop_duplicates(["order_id"]) 
    .withColumn("order_approved_at", f.col("order_approved_at").cast("timestamp"))
    .withColumn("order_purchase_timestamp", f.col("order_purchase_timestamp").cast("timestamp"))
    .withColumn("order_delivered_carrier_date", f.col("order_delivered_carrier_date").cast("timestamp"))
    .withColumn("order_delivered_customer_date", f.col("order_delivered_customer_date").cast("timestamp"))
    .withColumn("order_estimated_delivery_date", f.col("order_estimated_delivery_date").cast("timestamp"))
    .withColumn("order_status", f.initcap(f.col("order_status")))
    .withColumn("silver_ingested_at", f.current_timestamp())
    .withColumn("silver_updated_at", f.current_timestamp())
    .drop("ingestion_time")
)

# 3. view the final result - test before loading then comment out
# display(df_ord_cleanup)

# 4. Create table if not already exists
if not spark.catalog.tableExists("Silver.olist_orders"):
    df_ord_cleanup.write.format("delta").saveAsTable("Silver.olist_orders")



In [ ]:
# olist order table merge

silver_ord_tbl = DeltaTable.forName(spark, "Silver.olist_orders")

silver_ord_tbl.alias("target") \
  .merge (
    source = df_ord_cleanup.alias("source") ,
    condition = "target.order_id = source.order_id" 
                
          ) \
  .whenMatchedUpdate(
   set = {
        "order_status" : "source.order_status",
        "order_approved_at" : "source.order_approved_at",
        "order_delivered_carrier_date" : "source.order_delivered_carrier_date",
        "order_delivered_customer_date" : "source.order_delivered_customer_date",
        "order_estimated_delivery_date" : "source.order_estimated_delivery_date",
        "silver_updated_at": f.current_timestamp()
   }
  ) \
  .whenNotMatchedInsertAll() \
  .execute()